# Refactored train/test split pipeline
This notebook orchestrates the original `train_test_split.ipynb` workflow using code extracted into the `src` package.


In [ ]:
import os
import sys
import numpy as np

# Allow imports from the project root
sys.path.insert(0, os.path.abspath('..'))

from src.data_prep import prepare_data
from src import regression
from src.pipeline import FEATURE_SETS, plot_feature_histograms, reset_output_dir, run_baseline_regressions
from src.model_comparison import run_model_comparison
from src.classification import run_classification_analysis

import matplotlib
matplotlib.use("Agg")

In [ ]:
if os.path.exists('outputs/paper_plots') is False:
    os.makedirs('outputs/paper_plots')

In [ ]:
# Configuration flags and parameters
RUN_BASELINE_MODELS = True
RUN_BASELINE_MIXED = False
RUN_BASELINE_DOMAIN_SHIFT = True
BASELINE_MIXED_LABEL = 'Mixed split'
BASELINE_DOMAIN_LABEL = 'E-INSPIRE→INSPIRE'
BASELINE_PLOTTING = False
RANDOM_STATES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 1000000]

RUN_HISTOGRAMS = True
RUN_CORNER = False
RUN_CORNER_BY_DATASET = True
RUN_RESIDUALS = False
RUN_CLASSIFICATION = True
RUN_MODEL_COMPARISON = True

MODEL_COMPARISON_SEEDS = [1,2,3,4,5]
MODEL_COMPARISON_MODES = [
    # ('mixed', 'Mixed split'),
    ('domain_shift', 'E-INSPIRE→INSPIRE'),
]
CLASSIFICATION_THRESHOLDS = np.arange(0.25, 0.66, 0.05)
CLASSIFICATION_FEATURES = ['tau', 'lin_age_err', 'met', 'met_err', 'logM', 'rad_kpc']
CLASSIFICATION_DATASET_MODES = [
    # ('mixed', 'Mixed split'),
    ('domain_shift', 'E-INSPIRE-INSPIRE'),
]

CORNER_FEATURES = ['met', 'tau', 'met_err', 'lin_age_err', 'logM', 'rad_kpc', 'MgFe', 'vdisp', 'DoR']
CORNER_TARGET_DISPLAY_NAME = r'$\mathrm{DoR}$'

RESIDUAL_FEATURES = ['tau', 'met']
RESIDUAL_MODEL_PARAMS = {
    'max_depth': 8,
    'max_features': 0.8,
    'max_samples': 0.7,
    'min_samples_leaf': 3,
    'min_samples_split': 5,
    'n_estimators': 50,
    'random_state': 42,
}


In [ ]:
columns = ['vdisp','tau','MgFe', 'met_err', 'lin_age_err','met','rad_kpc','logM','DoR']
mixed_train_df, mixed_test_df = prepare_data(columns, restricted=False, pc=False, mix_datasets=True)
domain_train_df, domain_test_df = prepare_data(columns, restricted=False, pc=False, mix_datasets=False)

# Default references for downstream plots (mixed split)
train_df, test_df = mixed_train_df, mixed_test_df
regression.train_df = train_df
regression.test_df = test_df

# Unmixed data for E-INSPIRE vs INSPIRE visualisations
hist_train_df, hist_test_df = domain_train_df, domain_test_df


In [ ]:
if RUN_HISTOGRAMS:
    if RUN_BASELINE_DOMAIN_SHIFT:
        # Unmixed E-INSPIRE vs INSPIRE
        plot_feature_histograms(hist_train_df, hist_test_df, tag='unmixed')
    if RUN_BASELINE_MIXED:
        # Mixed train/test split
        plot_feature_histograms(mixed_train_df, mixed_test_df, tag='mixed')


In [ ]:
print(min(domain_test_df['tau']))

In [ ]:
grid = False
if grid:
    # Grid test thing here
    import itertools
    import numpy as np
    from sklearn.model_selection import KFold
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import SVR
    from sklearn.metrics import r2_score

    # Use the mixed train data prepared above
    feature_list = ['met', 'tau', 'met_err', 'lin_age_err', 'logM', 'rad_kpc', 'MgFe', 'vdisp']
    X = train_df[feature_list].to_numpy()
    y = train_df['DoR'].to_numpy()

    # Expanded grid across kernels and their relevant params
    C_values = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0]
    epsilon_values = [0.001, 0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.15]
    gamma_values = ['scale', 'auto', 0.01, 0.05, 0.1, 0.2]
    degree_values = [2, 3, 4]
    coef0_values = [0.0, 0.1, 0.5]
    kernels = ['rbf', 'linear', 'poly', 'sigmoid']

    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    def eval_config(config):
        # Build model with config fields
        model = SVR(**config)
        fold_scores = []
        for train_idx, val_idx in cv.split(X):
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X[train_idx])
            X_val_scaled = scaler.transform(X[val_idx])
            model.fit(X_train_scaled, y[train_idx])
            preds = model.predict(X_val_scaled)
            fold_scores.append(r2_score(y[val_idx], preds))
        return float(np.mean(fold_scores))

    configs = []
    for kernel in kernels:
        for C in C_values:
            for eps in epsilon_values:
                if kernel == 'rbf':
                    for gamma in gamma_values:
                        configs.append({'kernel': kernel, 'C': C, 'epsilon': eps, 'gamma': gamma})
                elif kernel == 'linear':
                    configs.append({'kernel': kernel, 'C': C, 'epsilon': eps})
                elif kernel == 'poly':
                    for degree in degree_values:
                        for gamma in gamma_values:
                            for coef0 in coef0_values:
                                configs.append({
                                    'kernel': kernel, 'C': C, 'epsilon': eps,
                                    'degree': degree, 'gamma': gamma, 'coef0': coef0
                                })
                elif kernel == 'sigmoid':
                    for gamma in gamma_values:
                        for coef0 in coef0_values:
                            configs.append({'kernel': kernel, 'C': C, 'epsilon': eps, 'gamma': gamma, 'coef0': coef0})

    results = []
    for idx, cfg in enumerate(configs, 1):
        score = eval_config(cfg)
        results.append((cfg, score))
        if idx % 100 == 0:
            print(f'Evaluated {idx}/{len(configs)} configs...')

    results = sorted(results, key=lambda x: x[1], reverse=True)
    print('Top grid combos (kernel, C, epsilon, gamma, degree, coef0 -> mean R^2):')
    for cfg, score in results[:15]:
        print(cfg, '->', f'{score:.4f}')


In [ ]:
# Prepare output directory used throughout the pipeline
reset_output_dir('outputs/tests')


In [ ]:
if RUN_BASELINE_MODELS:
    if RUN_BASELINE_MIXED:
        run_baseline_regressions(mixed_train_df, mixed_test_df, RANDOM_STATES, plotting=BASELINE_PLOTTING, tag=BASELINE_MIXED_LABEL)
        ensemble_mixed = regression.calculate_ensemble_metrics(
            dataset_frames={
            BASELINE_MIXED_LABEL: (mixed_train_df, mixed_test_df),
            BASELINE_DOMAIN_LABEL: (domain_train_df, domain_test_df),
            },
            filter_tags=[BASELINE_MIXED_LABEL],
            output_path='outputs/paper_plots/ensemble_results_mixed.csv',
        )
    if RUN_BASELINE_DOMAIN_SHIFT:
        run_baseline_regressions(domain_train_df, domain_test_df, RANDOM_STATES, plotting=BASELINE_PLOTTING, tag=BASELINE_DOMAIN_LABEL)
        regression.print_results_summary()
        ensemble_domain = regression.calculate_ensemble_metrics(
            dataset_frames={
                BASELINE_MIXED_LABEL: (mixed_train_df, mixed_test_df),
                BASELINE_DOMAIN_LABEL: (domain_train_df, domain_test_df),
            },
            filter_tags=[BASELINE_DOMAIN_LABEL],
            output_path='outputs/paper_plots/ensemble_results_domain_shift.csv',
        )


In [ ]:
if RUN_CORNER:
    regression.create_corner_plots(
        hist_test_df,
        CORNER_FEATURES,
        target='DoR',
        target_display_name=CORNER_TARGET_DISPLAY_NAME,
    )
    regression.create_corner_plots(
        hist_train_df,
        CORNER_FEATURES,
        target='DoR',
        target_display_name=CORNER_TARGET_DISPLAY_NAME,
    )


In [ ]:
if RUN_CORNER_BY_DATASET:
    feature_display_names = {
        'met': r'$\mathrm{[M/H] \, (dex)}$',
        'tau': r'$\tau_{\rm rel}$',
        'met_err': r'$\Delta\mathrm{[M/H] \, (dex)}$',
        'lin_age_err': r'$\Delta\mathrm{Age \, (Gyr)}$',
        'logM': r'$\log(M_{\star}/M_{\odot})$',
        'rad_kpc': r'$R_{\rm e} \, \mathrm{(kpc)}$',
        'MgFe': r'$\mathrm{[Mg/Fe] \, (dex)}$',
        'vdisp': r'$\sigma_{\star} \, \mathrm{(km/s)}$',
        'DoR': r'$\mathrm{DoR}$',
    }
    regression.plot_features_by_dataset(
        hist_train_df,
        hist_test_df,
        CORNER_FEATURES,
        feature_display_names=feature_display_names,
    )


In [ ]:
if RUN_RESIDUALS:
    regression.create_residual_analysis_report(
        train_df,
        test_df,
        RESIDUAL_FEATURES,
        'DoR',
        RESIDUAL_MODEL_PARAMS,
    )


In [ ]:
if RUN_CLASSIFICATION:
    for dataset_mode, dataset_label in CLASSIFICATION_DATASET_MODES:
        cv_results = run_classification_analysis(CLASSIFICATION_THRESHOLDS, CLASSIFICATION_FEATURES, dataset_mode=dataset_mode, dataset_label=dataset_label)


In [ ]:
MODEL_COMPARISON_SEEDS = [3,4,5,6,7]
if RUN_MODEL_COMPARISON:
    for dataset_mode, dataset_label in MODEL_COMPARISON_MODES:
        results_df, agg_results = run_model_comparison(MODEL_COMPARISON_SEEDS, True, dataset_mode=dataset_mode, dataset_label=dataset_label)
    print('Model comparison completed and results saved to outputs/tests/')


In [ ]:
# Permutation Importance for SVR
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import KFold

# Feature display names for plotting
feature_display_names = {
    'met': r'$\mathrm{[M/H]}$',
    'tau': r'$\mathrm{\tau_{\rm rel}}$',
    'met_err': r'$\Delta{\mathrm{[M/H]}}$',
    'lin_age_err': r'$\Delta{\mathrm{Age}}$',
    'logM': r'$\log(M/M_{\odot})$',
    'rad_kpc': r'$R \, \mathrm{(kpc)}$',
    'MgFe': r'$\mathrm{[Mg/Fe]}$',
    'vdisp': r'$\sigma_{\star} \, \mathrm{(km/s)}$',
}

# SVR parameters (best from grid search)
SVR_PARAMS = {
    'C': 5.0,
    'epsilon': 0.03,
    'kernel': 'poly',
    'gamma': 0.01,
    'degree': 3,
    'coef0': 0.5,
}

# Use domain shift data (E-INSPIRE -> INSPIRE)
# Complete set used for permutation importance
features = ['met', 'tau', 'met_err', 'lin_age_err', 'logM', 'rad_kpc', 'MgFe', 'vdisp']

X_train = domain_train_df[features]
y_train = domain_train_df['DoR']
X_test = domain_test_df[features]
y_test = domain_test_df['DoR']

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train SVR
model = SVR(**SVR_PARAMS)
model.fit(X_train_scaled, y_train)

# Calculate permutation importance on test set
print("Calculating permutation importance (this may take a moment)...")
result = permutation_importance(
    model, X_test_scaled, y_test,
    n_repeats=30,
    random_state=42,
    scoring='r2'
)

# Create results DataFrame
importance_df = pd.DataFrame({
    'feature': features,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std,
    'display_name': [feature_display_names[f] for f in features]
}).sort_values('importance_mean', ascending=True)

print("\nPermutation Importance (Test Set):")
print("=" * 60)
for _, row in importance_df.iterrows():
    print(f"{row['feature']:15s}: {row['importance_mean']:.4f} ± {row['importance_std']:.4f}")

# Plot
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Computer Modern",
    "figure.dpi": 300,
    "font.size": 20,
})

fig, ax = plt.subplots(figsize=(10, 6))

y_pos = np.arange(len(importance_df))
ax.barh(y_pos, importance_df['importance_mean'], xerr=importance_df['importance_std'],
        align='center', alpha=0.8, color='steelblue', ecolor='black', capsize=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(importance_df['display_name'], fontsize=20)
ax.set_xlabel(r'Permutation Importance', fontsize=22)
ax.set_title(r'SVR Permutation Importance (E-INSPIRE to INSPIRE)', fontsize=20)

# Add vertical line at 0
ax.axvline(x=0, color='red', linestyle='--', linewidth=1, alpha=0.7)

plt.tight_layout()
plt.savefig('outputs/paper_plots/svr_permutation_importance.pdf', bbox_inches='tight')
plt.show()

print(f"\nPlot saved to outputs/paper_plots/svr_permutation_importance.pdf")

# Extreme relic recognition - best and worst models at threshold 0.6
print("\nExtreme relic recognition at DoR >= 0.6 (test set):")
print("=" * 65)

relic_models = {
    'Best  (Stel. pop., alpha-abundance and kinematics)': ['met', 'tau', 'met_err', 'lin_age_err', 'MgFe', 'vdisp'],
    'Worst (Stel. pop. and structural / Euclid-like)':    ['met', 'tau', 'met_err', 'lin_age_err', 'logM', 'rad_kpc'],
}

y_true_binary = (y_test >= 0.6)
for label, feat_set in relic_models.items():
    sc = StandardScaler()
    Xtr = sc.fit_transform(domain_train_df[feat_set])
    Xte = sc.transform(domain_test_df[feat_set])
    m = SVR(**SVR_PARAMS)
    m.fit(Xtr, y_train)
    y_pred_binary = (m.predict(Xte) >= 0.6)
    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall    = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    n_true = y_true_binary.sum()
    n_pred = y_pred_binary.sum()
    print(f"{label}")
    print(f"  precision={precision:.3f}, recall={recall:.3f}  "
          f"(true: {n_true}, predicted: {n_pred})")
    print()

In [ ]:
# SVC Feature Importance Trends Across DoR Thresholds (Euclid-like features)
# Train a NEW classifier at each threshold, measure permutation importance
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

# Feature display names
feature_display_names = {
    'met': r'$\mathrm{[M/H]}$',
    'tau': r'$\tau_{\rm rel}$',
    'met_err': r'$\Delta\mathrm{[M/H]}$',
    'lin_age_err': r'$\Delta\mathrm{Age}$',
    'logM': r'$\log(M_{\star}/M_{\odot})$',
    'rad_kpc': r'$R_{\rm e}$',
}

# SVC parameters
SVC_PARAMS = {
    'C': 1.0,
    'kernel': 'rbf',
    'gamma': 'scale',
    'class_weight': 'balanced',
    'random_state': 42,
}

# Euclid-like features only
features = ['met', 'tau', 'met_err', 'lin_age_err', 'logM', 'rad_kpc']

# Use domain shift data
X_train = domain_train_df[features].values
y_train_continuous = domain_train_df['DoR'].values
X_test = domain_test_df[features].values
y_test_continuous = domain_test_df['DoR'].values

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Thresholds to analyze
thresholds = np.arange(0.30, 0.66, 0.05)

# Store results
importance_by_threshold = {}

print("Training SVC classifiers at each threshold (Euclid-like features)...")
print("=" * 70)

for threshold in thresholds:
    # Create binary labels
    y_train_binary = (y_train_continuous >= threshold).astype(int)
    y_test_binary = (y_test_continuous >= threshold).astype(int)
    
    # Check class balance
    pct_high = 100 * y_train_binary.mean()
    
    # Train classifier
    clf = SVC(**SVC_PARAMS)
    clf.fit(X_train_scaled, y_train_binary)
    
    # Evaluate
    y_pred = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test_binary, y_pred)
    f1 = f1_score(y_test_binary, y_pred)
    
    # Calculate permutation importance on test set
    result = permutation_importance(
        clf, X_test_scaled, y_test_binary,
        n_repeats=30,
        random_state=42,
        scoring='accuracy'
    )
    
    importance_by_threshold[threshold] = {
        'means': result.importances_mean,
        'stds': result.importances_std
    }
    
    print(f"Threshold {threshold:.2f}: {pct_high:.1f}% high class | Acc: {acc:.3f} | F1: {f1:.3f}")

# Convert to DataFrame for plotting
plot_data = []
for threshold, data in importance_by_threshold.items():
    for i, feat in enumerate(features):
        plot_data.append({
            'threshold': threshold,
            'feature': feat,
            'importance': data['means'][i],
            'std': data['stds'][i],
            'display_name': feature_display_names[feat]
        })

importance_df = pd.DataFrame(plot_data)

# Plot feature importance trends
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Computer Modern",
    "figure.dpi": 300,
    "font.size": 20,
})

fig, ax = plt.subplots(figsize=(10, 6))

for feat in features:
    feat_data = importance_df[importance_df['feature'] == feat]
    display_name = feature_display_names[feat]
    line, = ax.plot(feat_data['threshold'], feat_data['importance'],
                    'o-', label=display_name, linewidth=2, markersize=6)
    ax.errorbar(feat_data['threshold'], feat_data['importance'],
                yerr=feat_data['std'],
                fmt='none', ecolor=line.get_color(),
                alpha=0.3, capsize=3, elinewidth=4, capthick=4)

ax.set_xlabel('DoR Classification Threshold', fontsize=20)
ax.set_ylabel('Permutation Importance', fontsize=20)
ax.set_ylim(top=0.33)
ax.legend(loc='upper right', fontsize=17)
ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xticks(thresholds)
ax.set_xticklabels([f'{t:.2f}' for t in thresholds])

plt.tight_layout()
plt.savefig('outputs/paper_plots/svc_feature_importance_trends_euclid.pdf', bbox_inches='tight')
plt.show()

# Print summary table
print("\n" + "=" * 90)
print("Feature Importance by Threshold (Euclid-like):")
print("=" * 90)
header = f"{'Feature':<12}"
for t in thresholds:
    header += f"{t:.2f}".center(12)
print(header)
print("-" * 90)

for feat in features:
    row = f"{feat:<12}"
    for t in thresholds:
        val = importance_df[(importance_df['feature'] == feat) & (importance_df['threshold'] == t)]['importance'].values[0]
        row += f"{val:>10.4f}  "
    print(row)



# Print std table
print("\n" + "=" * 90)
print("Feature Importance STD by Threshold (Euclid-like):")
print("=" * 90)
print(header)
print("-" * 90)

for feat in features:
    row = f"{feat:<12}"
    for t in thresholds:
        val = importance_df[(importance_df['feature'] == feat) & (importance_df['threshold'] == t)]['std'].values[0]
        row += f"{val:>10.4f}  "
    print(row)
print(f"\nPlot saved to outputs/paper_plots/svc_feature_importance_trends_euclid.pdf")

In [ ]:
def c_monthly(years, x0,
              nominal_return=0.11, inflation=0.04,
              contrib_start=20, contrib_end=60):
    # Annual real factor (NOT "nominal - inflation", but ratio of factors)
    annual_real_factor = (1 + nominal_return) / (1 + inflation)

    # Convert annual factor to monthly factor
    monthly_factor = annual_real_factor ** (1/12)

    months = years * 12
    x = float(x0)
    z = float(x0)

    for m in range(months):
        # Growth over the month
        x *= monthly_factor
        z *= monthly_factor

        # Linear ramp of ANNUAL contribution, then convert to monthly
        t = m / (months - 1) if months > 1 else 0.0
        annual_contrib = contrib_start + t * (contrib_end - contrib_start)
        x += annual_contrib / 12.0  # paid monthly

    return x, z

print(c_monthly(20, 150))
